# 🧾 Doc Agent — Multi-Modal Document Intelligence
### Capstone Project | AI-Powered Invoice & Document Processing

---

## What is Doc Agent?

Doc Agent is a **LangGraph multi-agent AI system** that reads any business document
(invoice, receipt, PO, form, bank statement…), extracts ALL visible fields using
a vision-language model, validates data, auto-corrects errors, and flags uncertain
documents for human review.

### Pipeline Architecture


---

## ✅ Prerequisites

| Requirement | Details | Cost |
|-------------|---------|------|
| Google Account | For Colab + Gemini API | **Free** |
| Gemini API Key | 2 min setup (see Step 2) | **Free** |
| GPU / CUDA | Not required | — |

> 💡 Free Gemini tier: **30 requests/min · 1,500 requests/day**  
> ⚠️ **Each team member needs their own API key** — sharing one hits limits faster.

---

## 🔑 Get your free Gemini API Key (2 min)

1. Go to **https://aistudio.google.com/app/apikey**
2. Sign in with the same Google account you use for Colab
3. Click **"Create API key"** → **"Create API key in new project"**
4. Copy the key (starts with )
5. Add to **Colab Secrets** 🔑 (left sidebar) as   OR paste when prompted below

---

## ⚡ Quick Start



---
# 🛠️ STEP 1 — Setup  *(run once per Colab session)*


In [ ]:
# 1a. System dependencies (Tesseract OCR + Poppler)
import subprocess
print("Installing system dependencies...")
subprocess.run(["apt-get","update","-qq"], check=True, capture_output=True)
subprocess.run(["apt-get","install","-y","-qq",
                "tesseract-ocr","poppler-utils","libgl1","git"],
               check=True, capture_output=True)
r = subprocess.run(["tesseract","--version"], capture_output=True, text=True)
print("✅ Tesseract:", r.stderr.splitlines()[0] if r.stderr else "installed")
print("✅ Poppler: installed")


In [ ]:
# 1b. Clone project from GitHub
import os, subprocess
GITHUB_REPO = "https://github.com/yb-snow/capstone.git"
PROJECT_DIR = "/content/capstone"

if os.path.exists(PROJECT_DIR):
    print("Already cloned. Pulling latest...")
    r = subprocess.run(["git","-C",PROJECT_DIR,"pull","--quiet"], capture_output=True, text=True)
    print(r.stdout or "Already up to date.")
else:
    print("Cloning Doc Agent...")
    subprocess.run(["git","clone","--quiet",GITHUB_REPO,PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)
r = subprocess.run(["git","log","--oneline","-3"], capture_output=True, text=True)
print(f"✅ {PROJECT_DIR}")
print("Recent commits:
" + r.stdout)


In [ ]:
# 1c. Install Python packages  (~3 min first time)
import subprocess
print("Installing packages...")
subprocess.run(["pip","install","-q","-r",f"{PROJECT_DIR}/requirements.txt"], check=True)
print("✅ Done.")


---
# 🔑 STEP 2 — Configure Your Gemini API Key

**Option A — Colab Secrets (recommended):**  
1. Click the **🔑 key icon** in the left sidebar  
2. Add secret: Name = , Value = your  key  
3. Toggle **"Notebook access"** ON  

**Option B** — type when prompted below


In [ ]:
import os, getpass, subprocess

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    if not GEMINI_API_KEY: raise ValueError("empty")
    print("✅ Loaded from Colab Secrets.")
except Exception:
    print("Enter key manually (get free key at aistudio.google.com/app/apikey)")
    GEMINI_API_KEY = getpass.getpass("Paste Gemini API key (AIza...): ")

if not GEMINI_API_KEY.startswith("AIza"):
    print("⚠️  Key should start with AIza — double-check.")
else:
    print(f"✅ Key OK: {GEMINI_API_KEY[:8]}...")

os.makedirs(f"{PROJECT_DIR}/data/samples", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/data/output",  exist_ok=True)

env = f"""VLM_BACKEND=gemini
GEMINI_API_KEY={GEMINI_API_KEY}
GEMINI_MODEL=gemini-flash-latest
SQLITE_PATH={PROJECT_DIR}/data/invoices.db
CHROMA_PERSIST_DIR={PROJECT_DIR}/data/chroma
OCR_DPI=300
TOTAL_TOLERANCE=0.01
FUZZY_VENDOR_THRESHOLD=80
MAX_CORRECTION_ATTEMPTS=2
EXTRACTION_CONF_THRESHOLD=0.70
AUTO_APPROVE_THRESHOLD=0.85
"""

with open(f"{PROJECT_DIR}/.env", "w") as f:
    f.write(env)
print("✅ .env written.")


---
# ✔️ STEP 3 — Verify Setup


In [ ]:
import importlib.util, subprocess as sp, os, sys
sys.path.insert(0, "/content/capstone")

checks = {
    "Tesseract":       sp.run(["tesseract","--version"], capture_output=True).returncode == 0,
    "Poppler":         sp.run(["pdfinfo","-v"],         capture_output=True).returncode == 0,
    "Streamlit":       importlib.util.find_spec("streamlit")          is not None,
    "LangGraph":       importlib.util.find_spec("langgraph")           is not None,
    "google-genai":    importlib.util.find_spec("google.genai")        is not None,
    "Pillow":          importlib.util.find_spec("PIL")                 is not None,
    "pytesseract":     importlib.util.find_spec("pytesseract")         is not None,
    "ChromaDB":        importlib.util.find_spec("chromadb")            is not None,
    "rapidfuzz":       importlib.util.find_spec("rapidfuzz")           is not None,
    "OpenCV":          importlib.util.find_spec("cv2")                 is not None,
    "Pydantic v2":     importlib.util.find_spec("pydantic")            is not None,
    ".env file":       os.path.exists("/content/capstone/.env"),
    "app.py exists":   os.path.exists("/content/capstone/app.py"),
}

ok_all = True
for name, ok in checks.items():
    print(f"  {chr(9989) if ok else chr(10060)}  {name}")
    if not ok: ok_all = False

print()
print("🎉 All checks passed! Proceed to Step 4." if ok_all
      else "⚠️  Some checks failed. Re-run Step 1 cells.")


---
# 🚀 STEP 4 — Launch Doc Agent

Starts Streamlit and creates a **public Cloudflare Tunnel URL** (no account needed).

- Open the URL printed below
- **Login:**  / 
- The sidebar nav is always visible (no collapse button)

> ⏳ Wait ~20 seconds for the URL to appear


In [ ]:
import subprocess, time, re, sys, os
os.chdir("/content/capstone")

# Download cloudflared
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("Downloading Cloudflare tunnel...")
    subprocess.run(["wget","-q","-O","/usr/local/bin/cloudflared",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
        check=True)
    subprocess.run(["chmod","+x","/usr/local/bin/cloudflared"], check=True)

# Start Streamlit
streamlit_proc = subprocess.Popen(
    [sys.executable,"-m","streamlit","run","app.py",
     "--server.port","8501",
     "--server.headless","true",
     "--server.enableCORS","false",
     "--server.enableXsrfProtection","false"],
    cwd="/content/capstone",
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("⏳ Starting Streamlit...")
time.sleep(8)

# Open tunnel
tunnel_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared","tunnel","--url","http://localhost:8501"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE)

print("⏳ Creating public URL...")
public_url = None
for _ in range(40):
    line = tunnel_proc.stderr.readline().decode("utf-8", errors="ignore")
    m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", line)
    if m: public_url = m.group(); break
    time.sleep(1)

if public_url:
    pad = " " * max(0, 52 - len(public_url))
    print(f"""
╔══════════════════════════════════════════════════════════╗
║  🚀  Doc Agent is LIVE!                                  ║
║                                                          ║
║  {public_url}{pad}║
║                                                          ║
║  Login: demo / demo                                      ║
║  Keep this notebook tab open — closing stops the app     ║
╚══════════════════════════════════════════════════════════╝
""")
else:
    import urllib.request
    ip = urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode().strip()
    print(f"Tunnel failed. localtunnel password: {ip}")
    !npm install -g localtunnel --silent 2>/dev/null  # noqa
    !lt --port 8501  # noqa


---
# 📋 App Pages Guide

| Page | What to do |
|------|------------|
| 📊 **Dashboard** | KPIs, recent docs, pipeline health |
| 🔄 **Process Document** | Upload any PDF/image → AI extracts ALL fields |
| 👁️ **Review Queue** | Low-confidence docs await approval — see latency breakdown |
| 📋 **History** | All records · click row → ⏱ Latency tab shows per-stage timing |
| 🛠️ **Tech Stack** | Architecture + which library powers each block |
| ⚙️ **Settings** | Switch VLM (Gemini / Claude / Local MLX) live, no restart |

### Pipeline Latency Tracking
Every document processed shows timing per stage:



---
# ♻️ Restart App After Code Changes


In [ ]:
import subprocess, sys, time
subprocess.run(["pkill","-f","streamlit"], capture_output=True)
time.sleep(2)
subprocess.Popen(
    [sys.executable,"-m","streamlit","run","app.py",
     "--server.port","8501","--server.headless","true","--server.enableCORS","false"],
    cwd="/content/capstone", stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)
print("✅ Streamlit restarted. Refresh the URL from Step 4.")


---
# 🔬 OPTIONAL: Run Pipeline Cell by Cell
> Useful for learning how each stage works or for debugging.


In [ ]:
import sys, os
sys.path.insert(0, "/content/capstone")
os.chdir("/content/capstone")
print("✅ Project path ready")


In [ ]:
from google.colab import files
print("Upload a PDF, PNG, JPG, or TIFF document:")
uploaded = files.upload()
for fname in uploaded:
    DOC_PATH = f"/content/{fname}"
    with open(DOC_PATH, "wb") as f:
        f.write(uploaded[fname])
    print(f"✅ {DOC_PATH}  ({len(uploaded[fname])//1024} KB)")


In [ ]:
from pipeline.ingestion import load_document
import matplotlib.pyplot as plt

images = load_document(DOC_PATH)
print(f"✅ {len(images)} page(s), size: {images[0].size}")

fig, axes = plt.subplots(1, len(images), figsize=(8*len(images), 10))
if len(images) == 1: axes = [axes]
for i, (ax, img) in enumerate(zip(axes, images)):
    ax.imshow(img); ax.set_title(f"Page {i+1} (preprocessed)"); ax.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
from agents.extraction_agent import run as extract
from pipeline.ocr import ocr_extract_text
import time

ocr_text = ocr_extract_text(images[0])
print(f"OCR baseline: {len(ocr_text)} chars")

print("
Sending to Gemini (classify + extract in one call)...")
t0 = time.time()
result = extract(images[0], ocr_text)
elapsed = round(time.time() - t0, 2)

doc = result.extracted_data
print(f"
✅ Doc type   : {doc.doc_type.upper()}")
if doc.doc_subtype: print(f"   Subtype     : {doc.doc_subtype}")
print(f"   Confidence  : {result.confidence*100:.0f}%")
print(f"   Fields found: {len(doc.fields)}")
print(f"   Gemini time : {elapsed}s")
if doc.extraction_notes: print(f"   AI note     : {doc.extraction_notes}")
print("
Extracted fields:")
for k, v in doc.fields.items():
    if v is not None: print(f"  {k:<25}: {v}")


In [ ]:
import pandas as pd
from IPython.display import display
if doc.line_items:
    print(f"{len(doc.line_items)} line item(s):")
    display(pd.DataFrame(doc.line_items))
else:
    print("No line items found.")


In [ ]:
from agents.validation_agent import run as validate

validation = validate(result)
print(f"{chr(9989) if validation.is_valid else chr(10060)} Validation: {"PASSED" if validation.is_valid else "FAILED"}")
if validation.failed_fields: print(f"   Failed: {validation.failed_fields}")
print(f"   Confidence: {validation.overall_confidence*100:.0f}%")

rows = [{"Field":v.field,"Status":v.status.value,"Message":v.message or ""}
        for v in validation.field_validations]
display(pd.DataFrame(rows))


In [ ]:
from agents.correction_agent import run as correct

if not validation.is_valid:
    print(f"Auto-correcting: {validation.failed_fields}...")
    result = correct(images[0], result, validation)
    validation = validate(result)
    print(f"After correction: {chr(9989) if validation.is_valid else chr(9888)} {"PASSED" if validation.is_valid else "Still failing"}")
else:
    print("✅ No corrections needed.")


In [ ]:
import uuid, json
from database.storage import init_db, save_record, get_stats
from models.schemas import ProcessingRecord, ValidationStatus

init_db()
doc_id = str(uuid.uuid4())
status = ValidationStatus.VALID if validation.is_valid else ValidationStatus.PENDING

save_record(ProcessingRecord(
    document_id=doc_id, source_path=DOC_PATH,
    doc_type=doc.doc_type, raw_ocr_text=ocr_text,
    vlm_extraction={"fields":doc.fields,"line_items":doc.line_items},
    corrections_applied=[], final_data={"doc_type":doc.doc_type,"fields":doc.fields,"line_items":doc.line_items},
    validation_status=status, extraction_confidence=result.confidence,
    processing_notes=["Processed via Colab"],
))

stats = get_stats()
print(f"✅ Saved. ID: {doc_id}")
print(f"Database: {stats["total"]} docs  |  Success rate: {stats["success_rate"]}%")

# Download JSON
from google.colab import files as cfiles
export = {"doc_type":doc.doc_type, **doc.fields, "line_items":doc.line_items}
with open("/content/extracted.json","w") as f:
    json.dump(export, f, indent=2, default=str)
cfiles.download("/content/extracted.json")


---
# 🔁 OPTIONAL: Full Pipeline with Latency Tracking


In [ ]:
from graph.workflow import process_document_stream
import sys, os
sys.path.insert(0, "/content/capstone")
os.chdir("/content/capstone")

print(f"Running full pipeline on: {DOC_PATH}
")
print(f"{chr(39)}Stage{chr(39):<19} {chr(39)}Time{chr(39):>8}  Note")
print("-" * 65)

final_state = None
for node_name, state in process_document_stream(DOC_PATH):
    t = state.get("timings",{}).get(node_name,"")
    note = (state.get("processing_notes") or [""])[-1]
    print(f"  {node_name:<20} {str(t)+"s" if t else "":>8}  {note}")
    final_state = state

timings = final_state.get("timings",{})
total = timings.get("total", sum(v for k,v in timings.items() if k!="total"))
print("-" * 65)
print(f"  {"TOTAL":<20} {total:.2f}s")
print(f"
Status: {final_state["final_status"].value.upper()}")
ext = final_state.get("extraction")
if ext:
    print(f"Type: {ext.extracted_data.doc_type}  |  Fields: {len(ext.extracted_data.fields)}  |  Confidence: {ext.confidence*100:.0f}%")


---
# 👩‍💻 OPTIONAL: Contribute Changes to GitHub


In [ ]:
import subprocess, getpass
YOUR_NAME  = "Your Name"        # change this
YOUR_EMAIL = "you@example.com"  # change this
subprocess.run(["git","config","--global","user.name", YOUR_NAME],  check=True)
subprocess.run(["git","config","--global","user.email",YOUR_EMAIL], check=True)
print("✅ Git identity set.")


In [ ]:
COMMIT_MESSAGE = "Describe your changes"  # change this
TOKEN = getpass.getpass("GitHub Personal Access Token: ")
subprocess.run(["git","-C","/content/capstone","remote","set-url","origin",
                f"https://{TOKEN}@github.com/yb-snow/capstone.git"], check=True)
subprocess.run(["git","-C","/content/capstone","add","-A"],           check=True)
subprocess.run(["git","-C","/content/capstone","commit","-m",COMMIT_MESSAGE], check=True)
subprocess.run(["git","-C","/content/capstone","push"],               check=True)
subprocess.run(["git","-C","/content/capstone","remote","set-url","origin",
                "https://github.com/yb-snow/capstone.git"], check=True)
print("✅ Pushed.")


---
# 🛠️ Troubleshooting

| Problem | Solution |
|---------|----------|
| **429 Gemini rate limit** | Wait 60s. Free tier: 30 req/min, 1500/day. Use your own key. |
| **** | Re-run cell 1c (pip install) |
| **** | Re-run Step 2 |
| **Model not found (404)** | Try  or  in  |
| **Poppler/page count error** | Re-run cell 1a |
| **Tunnel URL never appears** | Wait 40s, then run localtunnel fallback |
| **Session expired** | Re-run ALL cells from Step 1 (free tier ~12h sessions) |
| **Login fails** | Credentials:  /  |
| **Sidebar not visible** | The nav cannot be collapsed — refresh the page. |

---

# 📖 Project Structure



**GitHub:** https://github.com/yb-snow/capstone  
**Login:** demo / demo  
